## 1. 研究動機 (Research Motivation)

建立此基準模型（Baseline Model）的核心目的，是為了提供客觀、可靠的對比標竿。

在導入複雜的深度增強式學習（Deep Reinforcement Learning, DRL）之前，必須先建立穩固的基礎：

1. **確保資料純淨度**：妥善處理歷史數據中的生存者偏誤（Survivorship Bias），確保回測結果貼近真實市場。

2. **建立傳統交易邏輯底盤**：實作具備共整合（Cointegration）與 Z-Score 邏輯的傳統配對交易系統，驗證均值回歸特性的有效性，作為後續 RL 代理人學習的基礎狀態與邏輯參考。

## 2. 文獻回顧 (Literature Review)

配對交易（Pairs Trading）為統計套利（Statistical Arbitrage）中經典的市場中性策略，本階段實作主要基於以下經典框架與最新研究進行建構：

- **傳統配對交易框架**：參考 Gatev, Goetzmann & Rouwenhorst (2006) 提出的經典方法論。該文獻奠定了使用距離法（Sum of Squared Differences, SSD）來篩選潛在配對，並採用滾動視窗（Rolling Window）機制的業界標準。
- **現代特徵擴展與機器學習應用**：參考 Yufei Sun (2025) 對於配對交易系統性文獻的最新整理。其研究確立了將總體經濟指標（Macroeconomic Indicators，例如 VIX）納入考量，以及後續應用機器學習進行動態分群（Clustering）的發展方向，這也成為我們後續強化學習模型的特徵工程藍圖。

## 3. 資料說明與前置處理 (Data Description & Preprocessing)

- **資料來源**：本研究使用 S&P 500 歷史日 K 線資料庫 (`sp500_data.db`)，資料期間涵蓋 **2000-01-01 至 2025-12-31** 的數據。
- **資料清洗步驟**：
    1. **前向填充與剔除稀疏股票**：針對價格序列進行前向填充（Forward-fill）最多 5 天，避免假日或停牌影響，並剔除有效交易日過少的稀疏股票。
    2. **生存者偏誤（Survivorship Bias）處理**：確保資料庫中包含歷史已下市成分股之數據，避免回測時因只考慮現有成分股而產生高估績效的生存者偏誤。
    3. **宏觀指標融合**：自動整合 Yahoo Finance 的 VIX 波動率指數至特徵矩陣中，為未來環境狀態空間（State Space）提供總體經濟環境的降維特徵。

## 4. 策略說明與實作細節 (Strategy Description & Implementation Details)

> **策略依據**：Gatev, Goetzmann & Rouwenhorst (2006)

### 策略總覽
配對交易為**市場中性**的統計套利策略。

| 階段 | 名稱 | 說明 |
|:---:|------|------|
| **1** | 初始化與資料收集 | 參數、SQLite、清理 |
| **2** | 配對選擇 | 協整檢驗 → SSD 排序 |
| **3** | 策略執行 | Z-Score 訊號 → PnL |
| **4** | 績效評估 | Sharpe、MDD、CAGR |


### 階段一：初始化與資料收集
### 1-1 套件安裝


In [1]:
import subprocess, sys
for pkg in ['statsmodels','plotly','ipywidgets','kaleido','yfinance']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
import warnings; warnings.filterwarnings('ignore')
import sqlite3, numpy as np, pandas as pd
import logging
from itertools import combinations
from statsmodels.tsa.stattools import coint
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML
pd.set_option('display.float_format','{:.4f}'.format)
import yfinance as yf
import time
import os
# print('套件導入完成。')

### 1-2 全域參數設定

> **使用者只需修改此區塊，即可調整所有策略參數。**

| 參數 | 預設値 | 說明 |
|------|--------|------|
| `DB_PATH` | `sp500_data.db` | SQLite 資料庫路徑 |
| `FORMATION_WINDOW` | `252` | 形成期（12M）|
| `TRADING_WINDOW` | `126` | 交易期（6M）|
| `ROLLING_WINDOW` | `20` | 滾動窗口（1M）|
| `COINT_P_VALUE` | `0.01` | 協整檢驗顯著水準 |
| `TOP_N_PAIRS` | `[5,20]` | Top-N 配對數 |
| `Z_ENTRY` | `2` | 進場閾値 |
| `Z_EXIT` | `0` | 出場閾値 |
| `MAX_LOSS_PCT` | `3.0` | 最大停損百分比 |
| `TRANSACTION_COST` | `0.0029` | 單邊成本 0.1% |
| `INITIAL_CAPITAL` | `10000` | 初始資金100000美金 |
| `CAPITAL_TRANCHES` | `7` | 分割成七等分 |


In [2]:
# ================================================================
#   Global Parameter Block  -- Edit here to tune strategy
# ================================================================
import os as _os
# DB_PATH: absolute path to the 342MB database (project root)
DB_PATH           = r'..\data\sp500_data.db'
START_DATE        = '2000-01-01'
END_DATE          = '2025-12-31'
FORMATION_WINDOW  = 252
TRADING_WINDOW    = 126
ROLLING_WINDOW    = 20
COINT_P_VALUE     = 0.01
TOP_N_PAIRS       = [5, 20]
Z_ENTRY           = 2
Z_EXIT            = 0
MAX_LOSS_PCT      = 3
TRANSACTION_COST  = 0.0029
SECTOR_NEUTRAL    = True
TARGET_SECTOR     = 'All'          # 設定為All為全產業，可指定特定產業
MIN_HISTORY_DAYS  = 200
HEDGE_RATIO       = 1.0
INITIAL_CAPITAL = 7000.0
CAPITAL_TRANCHES = 7
USE_DYNAMIC_SECTORS = True  # 開關：是否使用 API 補齊 Unknown 產業分類 (設為 False 則作為對照組)
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv' # 補齊後的分類快取儲存路徑
print(f'DB     : {DB_PATH}')
print(f'DB exists: {_os.path.exists(DB_PATH)}')
print(f'DB size  : {_os.path.getsize(DB_PATH)//1024//1024} MB')
print(f'Period : {START_DATE} ~ {END_DATE}')
print(f'Windows: {FORMATION_WINDOW}d / {TRADING_WINDOW}d')


DB     : ..\data\sp500_data.db
DB exists: True
DB size  : 342 MB
Period : 2000-01-01 ~ 2025-12-31
Windows: 252d / 126d


### 1-3 資料讀取與清理

**清理流程**: 前向填充(最多5天) → 剪除稀疏股票 → 對齊日期

**資料庫結構 (`sp500_data.db`)**:

| 資料表 | 關鍵欄位 |
|--------|----------|
| `daily_prices` | `ticker, date, adj_close` |
| `tickers` | `ticker, sector` |


In [3]:
def load_data_from_db(db_path, start_date, end_date):
    """Load price and sector data from SQLite."""
    conn = sqlite3.connect(db_path)
    price_queries = [
        (f"SELECT date, ticker, adj_close AS close FROM daily_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
        (f"SELECT date, ticker, close FROM stock_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
    ]
    prices_df = None
    for q in price_queries:
        try:
            prices_df = pd.read_sql_query(q, conn, parse_dates=['date'])
            if len(prices_df) > 0:
                print(f'Price rows: {len(prices_df):,}')
                break
        except Exception:
            continue
    if prices_df is None or len(prices_df) == 0:
        tbls = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
        conn.close()
        raise RuntimeError(f"No price table. Tables: {tbls['name'].tolist()}")
    sector_queries = [
        "SELECT ticker, sector FROM tickers",
        "SELECT ticker, sector FROM sp500_components GROUP BY ticker",
    ]
    sector_df = None
    for q in sector_queries:
        try:
            sector_df = pd.read_sql_query(q, conn)
            if len(sector_df) > 0:
                print(f'Sector rows: {len(sector_df):,}')
                break
        except Exception:
            continue
    conn.close()
    if sector_df is None or len(sector_df) == 0:
        print('WARNING: No sector table, using Unknown.')
        sector_df = pd.DataFrame({'ticker': prices_df['ticker'].unique(), 'sector': 'Unknown'})
        
    # [Mod] Mitigate Survivorship Bias: Check if delisted components exist
    max_date = prices_df['date'].max()
    max_dates = prices_df.groupby('ticker')['date'].max()
    delisted_count = (max_dates < max_date - pd.Timedelta(days=30)).sum()
    if delisted_count < 10:
        import logging
        logging.warning("STRICT RESEARCH LIMITATION: Database lacks historically delisted S&P 500 components. Survivorship Bias is present.")
        
    return prices_df, sector_df



def fix_unknown_sectors(sector_df, use_dynamic=True, save_path=r'data\imputed_sectors.csv'):
    """具備本機快取與全域開關控制的產業補齊模組"""
    # 1. 開關判斷：如果不使用動態補齊，直接原封不動回傳
    if not use_dynamic:
        print("【消融實驗設定】不使用動態產業補齊，維持原始 Unknown 分類作為對照組。")
        return sector_df

    # 確保儲存的目錄 (data\) 存在
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # 2. 快取讀取：如果已經抓過並存檔，直接載入
    if os.path.exists(save_path):
        print(f"從本機快取載入已補齊的產業分類: {save_path} ... (速度極快)")
        cached_df = pd.read_csv(save_path)
        update_df = cached_df.set_index('ticker')
        sector_df = sector_df.set_index('ticker')
        sector_df.update(update_df)
        return sector_df.reset_index()

    # 3. API 抓取：如果沒有快取，執行連線作業
    unknown_mask = sector_df['sector'] == 'Unknown'
    unknown_tickers = sector_df[unknown_mask]['ticker'].tolist()
    
    if not unknown_tickers:
        return sector_df

    print(f"找不到本機快取，正在透過 API 補齊 {len(unknown_tickers)} 檔 Unknown 股票的產業分類...")
    
    yf_logger = logging.getLogger('yfinance')
    original_level = yf_logger.level
    yf_logger.setLevel(logging.CRITICAL) 
    
    fixed_sectors = []
    
    for i, ticker in enumerate(unknown_tickers):
        try:
            info = yf.Ticker(ticker).info
            sector = info.get('sector', 'Unknown')
            fixed_sectors.append({'ticker': ticker, 'sector': sector})
            time.sleep(0.02) 
        except Exception:
            fixed_sectors.append({'ticker': ticker, 'sector': 'Unknown'})
            
        if (i + 1) % 50 == 0:
            print(f"已處理 {i + 1} / {len(unknown_tickers)}...")
            
    yf_logger.setLevel(original_level)
    
    # 4. 儲存快取：將剛抓下來的資料存成 CSV，下次就不用再抓了
    fetched_df = pd.DataFrame(fixed_sectors)
    fetched_df.to_csv(save_path, index=False)
    print(f"API 抓取完畢！已將動態產業分類永久儲存至: {save_path}")
    
    # 更新回原本的 DataFrame
    update_df = fetched_df.set_index('ticker')
    sector_df = sector_df.set_index('ticker')
    sector_df.update(update_df)
    sector_df = sector_df.reset_index()
    
    remaining = len(sector_df[sector_df['sector'] == 'Unknown'])
    print(f"補齊完成！剩餘真實無法識別(已下市)的 Unknown 股票數量: {remaining}")
    
    return sector_df

def preprocess_prices(prices_df, min_days=MIN_HISTORY_DAYS):
    """Pivot, forward-fill, drop sparse tickers."""
    pivot = prices_df.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
    pivot.index = pd.to_datetime(pivot.index)
    pivot.sort_index(inplace=True)
    pivot.ffill(limit=5, inplace=True)
    valid = pivot.columns[pivot.notna().sum() >= min_days]
    pivot = pivot[valid]
    print(f'Matrix: {len(pivot)} days x {len(pivot.columns)} tickers')

    # [Mod] Integrate Macroeconomic Indicator (VIX)
    import yfinance as yf
    print("Fetching VIX data...")
    try:
        vix_data = yf.download("^VIX", start=pivot.index.min(), end=pivot.index.max() + pd.Timedelta(days=1), progress=False)
        if isinstance(vix_data.columns, pd.MultiIndex):
            vix_close = vix_data['Close'].squeeze()
        else:
            vix_close = vix_data['Close']
        vix_df = pd.DataFrame({'VIX': vix_close})
        vix_df.index = pd.to_datetime(vix_df.index).tz_localize(None)
        vix_aligned = vix_df.reindex(pivot.index).ffill()
        print("VIX feature matrix aligned.")
    except Exception as e:
        print(f"Error fetching VIX: {e}")
        vix_aligned = pd.DataFrame(index=pivot.index, columns=['VIX'])

    return pivot, vix_aligned


In [4]:
# 1. 從資料庫載入原始資料
prices_raw, sector_info = load_data_from_db(DB_PATH, START_DATE, END_DATE)

# 2. 攔截並補齊 Unknown 產業分類 (傳入全域開關與路徑)
sector_info = fix_unknown_sectors(
    sector_info, 
    use_dynamic=USE_DYNAMIC_SECTORS, 
    save_path=IMPUTED_SECTOR_PATH
)

# 3. 執行原有的價格矩陣轉換與 VIX 融合
price_pivot, vix_features = preprocess_prices(prices_raw)
sector_map = sector_info.set_index('ticker')['sector'].to_dict()

# 4. 顯示結果
print(pd.Series(sector_map).value_counts().head(10))

Price rows: 3,607,121
Sector rows: 843
從本機快取載入已補齊的產業分類: ..\data\imputed_sectors.csv ... (速度極快)
Matrix: 6539 days x 660 tickers
Fetching VIX data...
VIX feature matrix aligned.
Unknown                   214
Industrials                94
Financials                 76
Information Technology     71
Health Care                60
Consumer Discretionary     48
Consumer Cyclical          37
Real Estate                36
Consumer Staples           36
Energy                     35
Name: count, dtype: int64


### 階段二：配對選擇（形成期）

**Engle-Granger 協整檢驗**：
$$P_{A,t} = \\alpha + \\beta P_{B,t} + \\varepsilon_t$$
如果殘差 $\\varepsilon_t$ 的 ADF $p < 0.01$，則判定協整。

**SSD 排序**: $\\text{SSD}(A,B)=\\sum(r_A^{norm}-r_B^{norm})^2$

SSD 越小 → 趨勢越相似 → 優先選入交易期。

**選定流程**: 同產業候選 → 協整檢驗 ($p<0.01$) → SSD由小到大排序 → Top-N


In [5]:
from joblib import Parallel, delayed
import pandas as pd
from itertools import combinations
from statsmodels.tsa.stattools import coint

def normalize_prices(prices):
    return prices / prices.iloc[0]

def compute_ssd(na, nb):
    return ((na - nb)**2).sum()

# --- 新增：獨立的單一配對檢驗邏輯 (用於多核心平行運算) ---
def test_single_coint(a, b, sector, price_window, coint_pval, min_len):
    """修復：實作雙向 Engle-Granger 協整檢驗以消弭非對稱性"""
    try:
        sa = price_window[a].dropna()
        sb = price_window[b].dropna()
        idx = sa.index.intersection(sb.index)
        if len(idx) < min_len:
            return None
            
        # 進行雙向檢定 (A對B, 以及 B對A)
        pval_ab = coint(sa[idx].values, sb[idx].values)[1]
        pval_ba = coint(sb[idx].values, sa[idx].values)[1]
        
        # 只要其中一個方向具備強共整合 (取最小值)，即視為有效配對
        min_pval = min(pval_ab, pval_ba)
        
        if min_pval < coint_pval:
            return {'stock_a': a, 'stock_b': b, 'sector': sector, 'coint_p': min_pval}
    except Exception:
        return None
    return None

# --- 修改：主篩選函數 ---
def select_pairs_for_window(price_window, sector_map, top_ns, 
                            coint_pval=COINT_P_VALUE, 
                            sector_neutral=SECTOR_NEUTRAL, 
                            target_sector=TARGET_SECTOR):
    """嚴格遵守文獻：先 Cointegration 檢驗 -> 後 SSD 排序，並徹底剔除 Unknown 產業"""
    
    norm = normalize_prices(price_window.dropna(axis=1))
    valid = norm.columns.tolist()
    
    # [關鍵修正]：在任何分群發生之前，直接將 Unknown 股票從有效候選池中剔除
    valid = [t for t in valid if sector_map.get(t, 'Unknown') != 'Unknown']
    
    # 產業過濾與分群邏輯
    if target_sector is not None and target_sector != 'All':
        # 如果指定了單一產業 (例如 'Information Technology')
        valid = [t for t in valid if sector_map.get(t, 'Unknown') == target_sector]
        candidates = [(a, b, target_sector) for a, b in combinations(valid, 2)]
        
    elif sector_neutral:
        # 產業中性：只有同產業的股票才能互相配對
        groups = {}
        for t in valid:
            sec = sector_map.get(t, 'Unknown')
            groups.setdefault(sec, []).append(t)
            
        candidates = [(a, b, s) for s, ms in groups.items() if len(ms) >= 2 for a, b in combinations(ms, 2)]
        
    else:
        # 全局對比 (Cross-Sector)：此時 Unknown 也已被提前剔除，不會干擾跨產業配對
        candidates = [(a, b, 'All') for a, b in combinations(valid, 2)]
        
    if not candidates:
        return {n: [] for n in top_ns}
        
    min_len = FORMATION_WINDOW * 0.8
    
    # 使用 JobLib 啟動多核心平行運算進行雙向協整檢驗
    coint_results = Parallel(n_jobs=-1, batch_size='auto')(
        delayed(test_single_coint)(a, b, sector, price_window, coint_pval, min_len)
        for a, b, sector in candidates
    )
    
    # 過濾掉未通過檢驗或回傳 None 的結果
    passed = [res for res in coint_results if res is not None]
    
    if not passed:
        return {n: [] for n in top_ns}
        
    # 對「通過協整檢驗」的配對計算 SSD
    for p in passed:
        p['ssd'] = compute_ssd(norm[p['stock_a']], norm[p['stock_b']])
        
    # 依照 SSD 排序並選出 Top-N
    df = pd.DataFrame(passed).sort_values('ssd')
    return {n: df.head(n).to_dict('records') for n in top_ns}

### 資金配置與部位控管 (Capital Allocation & Position Sizing)

為了解決滾動視窗（Rolling Window）產生的重疊交易期問題，本策略採用**獨立資金分群（Capital Tranches）**設計：

1. **資金分割 (Capital Tranches)**：
   - 設定預設初始總資金為 $10,000 (INITIAL_CAPITAL)。
   - 由於交易期 TRADING_WINDOW 為 126 天（約 6 個月），而滾動步長 ROLLING_WINDOW 為 20 天（約 1 個月），同一時間市場上最多會有 $\lceil 126 / 20 \rceil = 7$ 個交易視窗重疊。
   - 策略將總資金平分為 7 等分 (CAPITAL_TRANCHES = 7)，每等分約為 $1428.57。各個重疊視窗所分配到的資金獨立運作，互不干涉，確保每個月初都有固定比例的閒置資金可供新視窗使用。

2. **配對權重分配 (Equal Weighting per Pair)**：
   - 在單一交易視窗中，該視窗的專屬資金 ($1428.57) 會**等權重**分配給該期篩選出的 Top-N 組配對。
   - 例如：若設定 TOP_N = 5，則每組配對可分配到 $285.71 的操作資金。

3. **市場中性 (Dollar-Neutral)**：
   - 針對每組配對的操作，嚴格維持 **Dollar-Neutral** 原則 (HEDGE_RATIO = 1.0)。
   - 每次進場時，皆會依據目標訊號等值買進（Long $1）強勢股並賣空（Short $1）弱勢股，確保不受單一產業或大盤整體漲跌的影響。

### 階段三：策略執行（交易期）

$$Z_t = (\\text{spread}_t - \\mu_{hist})/\\sigma_{hist}$$

| 訊號 | 操作 |
|------|---------|
| $Z > +1.5$ | 放空A、做多B |
| $Z < -1.5$ | 做多A、放空B |
| $Z(絕對值) < 0.75$ | 平倉 |
| $Z(絕對值) > 3.0$ | 停損平倉 |

**跨日執行法 (Wait-One-Day)**: 訊號後隔日執行，避免 bid-ask bounce。
**Dollar-Neutral**: 每配對 $1 做多 + $1 放空。


In [6]:
# --- 請完全替換 Cell 6 ---

def execute_pair_trades(trade_prices, pair, form_prices, 
                        z_entry=Z_ENTRY, z_exit=Z_EXIT, 
                        max_loss_pct=MAX_LOSS_PCT, cost=TRANSACTION_COST, 
                        hedge_ratio=HEDGE_RATIO):
    a, b = pair['stock_a'], pair['stock_b']
    try:
        pa_t, pb_t = trade_prices[a].dropna(), trade_prices[b].dropna()
        pa_f, pb_f = form_prices[a].dropna(), form_prices[b].dropna()
    except KeyError:
        return pd.Series(0.0, index=trade_prices.index)
        
    common = pa_t.index.intersection(pb_t.index)
    fi = pa_f.index.intersection(pb_f.index)
    if len(common) < 5 or len(fi) < 5: 
        return pd.Series(0.0, index=trade_prices.index)
    
    base_a, base_b = pa_f[fi].iloc[0], pb_f[fi].iloc[0]
    
    # 計算形成期的歷史均值與標準差
    fs = (pa_f[fi] / base_a) - (pb_f[fi] / base_b)
    mu, sd = fs.mean(), fs.std()
    if sd < 1e-8: 
        return pd.Series(0.0, index=trade_prices.index)
    
    # 交易期正規化與 Z 值計算 (延續形成期的數學尺度)
    na = pa_t[common] / base_a
    nb = pb_t[common] / base_b
    z = ((na - nb) - mu) / sd
    
    z_vals, na_vals, nb_vals = z.values, na.values, nb.values
    
    actual_stop_loss = -abs(max_loss_pct) / 100.0 if max_loss_pct > 0 else max_loss_pct
    
    # 狀態追蹤變數
    pos = 0               # 今日實際倉位
    target_pos = 0        # 昨天的訊號決定的今日目標倉位 (修復前視偏誤)
    pna, pnb = na_vals[0], nb_vals[0]
    entry_pa, entry_pb = 0.0, 0.0
    in_cooldown = False
    pnl_d = {}

    for i in range(len(common)):
        zi, cna, cnb = z_vals[i], na_vals[i], nb_vals[i]
        dt = common[i]
        p = 0.0
        
        # 1. 計算「持倉」帶來的當日報酬 (從昨天到今天的價格變動)
        if i > 0 and pos != 0:
            p += pos * ((cna - pna)/pna - hedge_ratio * (cnb - pnb)/pnb)
            
        # 2. 執行昨日產生的訊號 (跨日執行：扣除手續費並記錄真實進場價)
        if target_pos != pos:
            p -= 2 * cost * (1 + hedge_ratio)
            if target_pos != 0:
                entry_pa, entry_pb = cna, cnb # 真實進場點為當日價格
            else:
                entry_pa, entry_pb = 0.0, 0.0
            pos = target_pos # 正式轉換倉位
            
        # 3. 計算盤中的未實現累積損益 (用於觸發明天的停損)
        unrealized_pnl = 0.0
        if pos != 0 and entry_pa > 0 and entry_pb > 0:
            unrealized_pnl = pos * ((cna - entry_pa)/entry_pa - hedge_ratio * (cnb - entry_pb)/entry_pb)
            
        # 4. 盤後產生「明日」的目標倉位訊號 (狀態機)
        if in_cooldown:
            if abs(zi) < z_exit:
                in_cooldown = False
        elif pos == 0:
            if zi > z_entry:
                target_pos = -1
            elif zi < -z_entry:
                target_pos = +1
        else:
            if unrealized_pnl <= actual_stop_loss:
                target_pos = 0
                in_cooldown = True # 觸發停損，進入冷卻
            elif abs(zi) < z_exit:
                target_pos = 0     # 正常停利平倉

        # 紀錄今天的收盤狀態，供明天迴圈使用
        pna, pnb = cna, cnb
        pnl_d[dt] = p
        
    return pd.Series(pnl_d, dtype=float).reindex(trade_prices.index, fill_value=0.0)

### 滾動視窗主迴圈


In [7]:
def run_backtesting(price_pivot, sector_map,
                    formation_window=FORMATION_WINDOW,
                    trading_window=TRADING_WINDOW,
                    rolling_window=ROLLING_WINDOW,
                    top_ns=TOP_N_PAIRS,
                    initial_cap=INITIAL_CAPITAL,
                    tranches=CAPITAL_TRANCHES):
    
    dates = price_pivot.index
    N = len(dates)
    all_pnl = {n: pd.DataFrame(index=dates) for n in top_ns}
    records = []
    si, wid = 0, 0
    
    # 每個視窗被分配到的絕對資金
    tranche_capital = initial_cap / tranches 
    
    print(f'Starting backtest... Initial Capital: ${initial_cap}, Allocated per Window: ${tranche_capital:.2f}')

    while si + formation_window + trading_window <= N:
        fe = si + formation_window
        te = fe + trading_window
        
        fp = price_pivot.iloc[si:fe]
        tp = price_pivot.iloc[fe:te]
        
        sel = select_pairs_for_window(fp, sector_map, top_ns)
        rec = {'window_id': wid, 'form_start': dates[si], 'form_end': dates[fe-1],
               'trade_start': dates[fe], 'trade_end': dates[te-1], 'pairs_by_n': sel}
        records.append(rec)
        
        # 用於終端機印出的追蹤字串
        track_str = f"Window {wid:02d} | Trade: {dates[fe].date()} ~ {dates[te-1].date()}"
        
        for n in top_ns:
            pairs = sel.get(n, [])
            col = f'W{wid:02d}'
            wpnl = pd.Series(0.0, index=dates)
            
            if pairs:
                # 將該視窗的資金均分給 Top-N 的每個配對
                pair_capital = tranche_capital / len(pairs) 
                
                for p in pairs:
                    # 取得單一配對的百分比報酬
                    pct_return = execute_pair_trades(tp, p, fp) 
                    # 轉換為絕對金額 (百分比 * 分配資金)
                    wpnl = wpnl.add(pct_return * pair_capital)
                    
            all_pnl[n][col] = wpnl
            
            # 針對特定 N (例如 Top-5) 印出該視窗結束時的資金變動與損益百分比
            if n == top_ns[0]: # 預設印出第一個 Top-N 組合的狀況
                window_abs_pnl = wpnl.sum()
                window_pct_return = (window_abs_pnl / tranche_capital) * 100
                track_str += f" | Top-{n} PnL: ${window_abs_pnl:+.2f} ({window_pct_return:+.2f}%)"
                
        print(track_str)
        si += rolling_window
        wid += 1
        
    print(f'Backtest done: {wid} windows.')
    return all_pnl, records

# 啟動回測並將結果賦值給全域變數 all_pnl 與 window_records
all_pnl, window_records = run_backtesting(
    price_pivot, 
    sector_map,
    formation_window=FORMATION_WINDOW,
    trading_window=TRADING_WINDOW,
    rolling_window=ROLLING_WINDOW,
    top_ns=TOP_N_PAIRS,
    initial_cap=INITIAL_CAPITAL, 
    tranches=CAPITAL_TRANCHES
)

Starting backtest... Initial Capital: $7000.0, Allocated per Window: $1000.00
Window 00 | Trade: 2001-01-02 ~ 2001-07-02 | Top-5 PnL: $-45.78 (-4.58%)
Window 01 | Trade: 2001-01-31 ~ 2001-07-31 | Top-5 PnL: $-64.98 (-6.50%)
Window 02 | Trade: 2001-03-01 ~ 2001-08-28 | Top-5 PnL: $-66.65 (-6.67%)
Window 03 | Trade: 2001-03-29 ~ 2001-10-02 | Top-5 PnL: $+2.68 (+0.27%)
Window 04 | Trade: 2001-04-27 ~ 2001-10-30 | Top-5 PnL: $-75.94 (-7.59%)
Window 05 | Trade: 2001-05-25 ~ 2001-11-28 | Top-5 PnL: $-69.57 (-6.96%)
Window 06 | Trade: 2001-06-25 ~ 2001-12-27 | Top-5 PnL: $-47.95 (-4.80%)
Window 07 | Trade: 2001-07-24 ~ 2002-01-28 | Top-5 PnL: $-61.71 (-6.17%)
Window 08 | Trade: 2001-08-21 ~ 2002-02-26 | Top-5 PnL: $-57.55 (-5.76%)
Window 09 | Trade: 2001-09-25 ~ 2002-03-26 | Top-5 PnL: $-41.44 (-4.14%)
Window 10 | Trade: 2001-10-23 ~ 2002-04-24 | Top-5 PnL: $+12.59 (+1.26%)
Window 11 | Trade: 2001-11-20 ~ 2002-05-22 | Top-5 PnL: $+108.60 (+10.86%)
Window 12 | Trade: 2001-12-19 ~ 2002-06-20 | 

### 階段四：績效評估與對比分析

| 指標 | 公式 |
|------|------|
| **CAGR** | $(1+r_{total})^{252/T}-1$ |
| **Sharpe** | $\\mu_{ann}/\\sigma_{ann}$ |
| **MDD** | $\\max(1-NAV_t/peak_t)$ |
| **Sortino** | 使用下行標準差 |


In [8]:
def compute_metrics(ret, label=''):
    """Annualised performance metrics from daily return series."""
    r = ret.dropna().replace([np.inf,-np.inf], np.nan).dropna()
    T = len(r)
    if T == 0: return {}
    cum    = (1+r).cumprod()
    tot    = cum.iloc[-1]-1
    cagr   = (1+tot)**(252/T)-1
    vol    = r.std()*np.sqrt(252)
    sharpe = (r.mean()*252)/vol if vol>0 else np.nan
    mdd    = ((cum-cum.cummax())/cum.cummax()).min()
    dn     = r[r<0].std()*np.sqrt(252)
    sortino= (r.mean()*252)/dn if dn>0 else np.nan
    return {'策略':label,'CAGR(%)':round(cagr*100,2),
            '年化波動(%)':round(vol*100,2),
            'Sharpe':round(sharpe,3) if not np.isnan(sharpe) else np.nan,
            'Sortino':round(sortino,3) if not np.isnan(sortino) else np.nan,
            '最大回撤(%)':round(mdd*100,2),
            '勝率(%)':round((r>0).mean()*100,2),
            '總報酬(%)':round(tot*100,2)}

strat_ret = {}
for n in TOP_N_PAIRS:
    df = all_pnl[n]
    if df.empty: continue
    
    # 將所有平行視窗的「絕對金額損益」加總
    portfolio_daily_pnl = df.sum(axis=1)
    
    # 除以總初始資金 (10000美金)，得到嚴格遵守資金控管的投資組合每日報酬率
    strat_ret[f'Top-{n}'] = portfolio_daily_pnl / INITIAL_CAPITAL
    
# strat_ret['S&P500(EW)'] = price_pivot.pct_change().mean(axis=1)
pct_change = price_pivot.pct_change()
# 將無限大替換為 NaN
pct_change = pct_change.replace([np.inf, -np.inf], np.nan)
# 排除單日暴漲超過 100% 的極端錯誤數據 (通常是除權息未還原導致)
pct_change = pct_change.where(pct_change < 1.0, np.nan) 

strat_ret['S&P500(EW)'] = pct_change.mean(axis=1)

metrics_df = pd.DataFrame(
    [m for lbl,r in strat_ret.items() for m in [compute_metrics(r,lbl)] if m]
).set_index('策略')
display(metrics_df.style.background_gradient(cmap='RdYlGn',axis=0))


,CAGR(%),年化波動(%),Sharpe,Sortino,最大回撤(%),勝率(%),總報酬(%)
策略,,,,,,,
Top-5,-3.380000,1.900000,-1.797000,-2.418000,-59.990000,43.460000,-59.060000
Top-20,-3.360000,1.230000,-2.779000,-3.842000,-59.070000,40.370000,-58.820000
S&P500(EW),11.900000,20.340000,0.655000,0.841000,-53.150000,53.920000,1748.000000


### 累積 NAV 曲線與回撤圖


In [9]:
COLORS = px.colors.qualitative.Plotly
fig_p = make_subplots(rows=2,cols=1,shared_xaxes=True,
    row_heights=[0.65,0.35],
    subplot_titles=['累積淨値（NAV）曲線','歷史回撤（Drawdown%）'],
    vertical_spacing=0.08)
for i,(lbl,ret) in enumerate(strat_ret.items()):
    r=ret.dropna(); nav=(1+r).cumprod()
    dd=((nav-nav.cummax())/nav.cummax())*100
    c=COLORS[i%len(COLORS)]
    fig_p.add_trace(go.Scatter(x=nav.index,y=nav.values,name=lbl,
        line=dict(color=c,width=2),
        hovertemplate=f'%{{x|%Y-%m-%d}}<br>NAV:%{{y:.4f}}<extra>{lbl}</extra>'),row=1,col=1)
    fig_p.add_trace(go.Scatter(x=dd.index,y=dd.values,name=lbl,showlegend=False,
        fill='tozeroy',line=dict(color=c,width=1),
        hovertemplate=f'%{{x|%Y-%m-%d}}<br>DD:%{{y:.2f}}%<extra>{lbl}</extra>'),row=2,col=1)
fig_p.update_layout(title='配對交易策略績效對比',height=650,template='plotly_dark',
    hovermode='x unified',legend=dict(orientation='h',y=1.02,x=1,xanchor='right'))
fig_p.show()


In [10]:
import os
import pandas as pd
import numpy as np
from collections import Counter

def export_performance_reports(all_pnl, window_records, top_n=5, 
                               initial_cap=INITIAL_CAPITAL, tranches=CAPITAL_TRANCHES,
                               output_dir=r'..\docs'):
    """將回測結果匯出為具備詳細統計指標、配對熱度與產業別的 CSV 報表"""
    
    os.makedirs(output_dir, exist_ok=True)
    allocated_cap = initial_cap / tranches
    
    # ==========================================
    # 1. 視窗層級報表 (Window-Level Report)
    # ==========================================
    win_data = []
    for rec in window_records:
        wid = rec['window_id']
        ts = rec['trade_start']
        te = rec['trade_end']
        pairs = rec['pairs_by_n'].get(top_n, [])
        
        # [新增] 萃取配對名稱與其對應的產業
        pair_details = [f"{p['stock_a']}-{p['stock_b']} [{p.get('sector', 'Unknown')}]" for p in pairs]
        
        # [新增] 統計該視窗涵蓋的所有產業 (去除重複)
        sectors_in_win = list(set([p.get('sector', 'Unknown') for p in pairs]))
        sector_str = ", ".join(sectors_in_win) if sectors_in_win else "無"
        
        col = f'W{wid:02d}'
        wpnl = all_pnl[top_n][col]
        
        trade_days = (te - ts).days
        if trade_days <= 0: trade_days = 1 
        
        win_pnl = wpnl.sum()
        win_ret = win_pnl / allocated_cap
        
        active_pnl = wpnl[(wpnl.index >= ts) & (wpnl.index <= te)]
        daily_ret = active_pnl / allocated_cap
        
        cagr = (1 + win_ret) ** (365 / trade_days) - 1 if win_ret > -1 else -1
        std = daily_ret.std()
        sharpe = (daily_ret.mean() * 252) / (std * np.sqrt(252)) if std > 1e-8 else 0
        
        win_data.append({
            '視窗代碼': f'W{wid:02d}',
            '交易起始日': ts.strftime('%Y-%m-%d'),
            '交易結束日': te.strftime('%Y-%m-%d'),
            '分配資金(USD)': round(allocated_cap, 2),
            '視窗總損益(USD)': round(win_pnl, 2),
            '視窗報酬率(%)': round(win_ret * 100, 2),
            '年化報酬率(CAGR %)': round(cagr * 100, 2),
            '夏普率(Sharpe)': round(sharpe, 3),
            '涵蓋產業': sector_str,                # <--- 新增欄位
            '入選配對數量': len(pairs),
            '配對清單 (含產業)': ", ".join(pair_details)  # <--- 強化欄位
        })
        
    df_window = pd.DataFrame(win_data)
    win_csv_path = os.path.join(output_dir, f'window_report_Top{top_n}.csv')
    df_window.to_csv(win_csv_path, index=False, encoding='utf-8-sig')
    
    # ==========================================
    # 2. 年度層級報表與配對/產業熱度 (Yearly Report)
    # ==========================================
    portfolio_daily_pnl = all_pnl[top_n].sum(axis=1)
    
    yearly_data = []
    years = portfolio_daily_pnl.index.year.unique()
    
    rolling_capital = initial_cap 
    
    for y in years:
        y_pnl_series = portfolio_daily_pnl[portfolio_daily_pnl.index.year == y]
        y_pnl = y_pnl_series.sum()
        
        start_cap = rolling_capital
        end_cap = start_cap + y_pnl
        
        y_ret = y_pnl / start_cap if start_cap > 0 else 0
        
        y_ret_series = y_pnl_series / start_cap if start_cap > 0 else y_pnl_series * 0
        
        std = y_ret_series.std()
        sharpe = (y_ret_series.mean() * 252) / (std * np.sqrt(252)) if std > 1e-8 else 0
        
        cum = (1 + y_ret_series).cumprod()
        peak = cum.cummax()
        dd = (cum - peak) / peak
        mdd = dd.min()
        
        cumulative_ret = (end_cap - initial_cap) / initial_cap
        
        # [新增] 年度配對與產業熱度邏輯
        y_pairs = []
        y_sectors = []
        for rec in window_records:
            ts, te = rec['trade_start'], rec['trade_end']
            if ts.year == y or te.year == y:
                pairs = rec['pairs_by_n'].get(top_n, [])
                y_pairs.extend([f"{p['stock_a']}-{p['stock_b']}" for p in pairs])
                y_sectors.extend([p.get('sector', 'Unknown') for p in pairs])
        
        heat_count = Counter(y_pairs).most_common(3)
        heat_str = ", ".join([f"{k} ({v}次)" for k, v in heat_count]) if heat_count else "無"
        
        # [新增] 統計當年度最活躍的產業
        sector_count = Counter(y_sectors).most_common(3)
        sector_str = ", ".join([f"{k} ({v}次)" for k, v in sector_count]) if sector_count else "無"
        
        yearly_data.append({
            '年度': y,
            '期初資金(USD)': round(start_cap, 2),
            '年度總損益(USD)': round(y_pnl, 2),
            '期末資金(USD)': round(end_cap, 2),
            '年度報酬率(%)': round(y_ret * 100, 2),
            '累積總報酬率(%)': round(cumulative_ret * 100, 2),
            '年度夏普率': round(sharpe, 3),
            '最大回撤(MDD %)': round(mdd * 100, 2),
            '年度熱門產業 (Top 3)': sector_str,   # <--- 新增欄位
            '年度配對熱度 (Top 3)': heat_str
        })
        
        rolling_capital = end_cap
        
    df_yearly = pd.DataFrame(yearly_data)
    yearly_csv_path = os.path.join(output_dir, f'yearly_report_Top{top_n}.csv')
    df_yearly.to_csv(yearly_csv_path, index=False, encoding='utf-8-sig')
    
    print(f"📊 報表輸出成功！")
    print(f"1. 視窗層級報表已儲存至: {win_csv_path}")
    print(f"2. 年度層級報表已儲存至: {yearly_csv_path}")
    
    return df_window, df_yearly

# 執行匯出
df_win, df_year = export_performance_reports(all_pnl, window_records, top_n=TOP_N_PAIRS[0])

# 在 Jupyter 畫面上預覽年度報表
display(df_year)

📊 報表輸出成功！
1. 視窗層級報表已儲存至: ..\docs\window_report_Top5.csv
2. 年度層級報表已儲存至: ..\docs\yearly_report_Top5.csv


,年度,期初資金(USD),年度總損益(USD),期末資金(USD),年度報酬率(%),累積總報酬率(%),年度夏普率,最大回撤(MDD %),年度熱門產業 (Top 3),年度配對熱度 (Top 3)
0,2000,7000.0000,0.0000,7000.0000,0.0000,0.0000,0.0000,0.0000,無,無
1,2001,7000.0000,-429.0600,6570.9400,-6.1300,-6.1300,-3.1790,-6.3000,"Real Estate (40次), Financials (7次), Industrial...","ARE-REG (4次), ARE-EQR (4次), AVB-BXP (3次)"
2,2002,6570.9400,-211.3200,6359.6200,-3.2200,-9.1500,-1.1520,-5.1700,"Real Estate (42次), Financials (22次), Industria...","ARE-EQR (3次), SLG-UDR (3次), REG-VNO (3次)"
3,2003,6359.6200,-321.9000,6037.7200,-5.0600,-13.7500,-2.3790,-5.6400,"Real Estate (34次), Financials (33次), Utilities...","HBAN-MTB (9次), CPT-EQR (6次), MTB-RF (4次)"
4,2004,6037.7200,45.4000,6083.1200,0.7500,-13.1000,0.2870,-1.7800,"Real Estate (28次), Utilities (26次), Financials...","HBAN-MTB (6次), NEE-SO (6次), BRK-B-SPGI (4次)"
5,2005,6083.1200,-239.1100,5844.0200,-3.9300,-16.5100,-1.5320,-4.3800,"Utilities (30次), Real Estate (28次), Financials...","NEE-SO (4次), ATO-NI (4次), FRT-SPG (4次)"
6,2006,5844.0200,-147.5900,5696.4300,-2.5300,-18.6200,-1.4050,-4.7400,"Real Estate (26次), Financials (21次), Communica...","GOOG-GOOGL (17次), EQR-MAA (5次), ERIE-KEY (3次)"
7,2007,5696.4300,-167.6500,5528.7800,-2.9400,-21.0200,-1.0780,-5.2500,"Financials (29次), Communication Services (20次)...","GOOG-GOOGL (18次), GIS-KMB (4次), ALL-MET (4次)"
8,2008,5528.7800,-643.8300,4884.9500,-11.6400,-30.2200,-3.5150,-11.2400,"Utilities (36次), Communication Services (20次),...","GOOG-GOOGL (19次), EVRG-XEL (5次), CMS-CNP (4次)"
9,2009,4884.9500,-397.3500,4487.6000,-8.1300,-35.8900,-2.6690,-9.6000,"Utilities (31次), Communication Services (19次),...","GOOG-GOOGL (19次), AVB-CPT (6次), EVRG-XEL (3次)"


## 5.未來工作

### 特徵工程與動態分群：
這個階段的目標是解決傳統靜態配對容易遭遇的「結構漂移」問題，建立具備抗震能力的動態配對池。

●擴充進階特徵：在現有的對數報酬率、波動率（如 ATR）與 VIX 總體經濟數據之上，加入能衡量均值回歸強度的進階特徵，例如 Hurst 指數、擴單根檢定（ADF Test）統計量與動態半衰期（Half-life） 。

●實施板塊過濾與分群：保留目前實作的「產業板塊過濾（Sector-based filtering）」作為前置處理 。

●接著，導入 Scikit-learn 的 DBSCAN 或 HDBSCAN 進行無監督學習，這類演算法能自動將財報爆雷的「雜訊股票」剔除 。

●動態滾動選股：以「月」為單位執行滾動式動態分群，並從中嚴格挑選出 1 至 3 組具備高度共整合特徵的經典配對，準備餵給強化學習代理人 。

### 強化學習模型建構
此階段將運用深度強化學習（DRL），取代傳統固定的 Z-Score 門檻，實現動態進出場控制。

●定義馬可夫決策過程 (MDP)：狀態空間 (State)：整合 VIX 指數、標準化價差與動態半衰期，賦予模型總體市場風險感知能力 。

●動作空間 (Action)：設定為離散空間（做多價差、放空價差、平倉/空手）以加速收斂 。

●獎勵函數 (Reward)：設計扣除交易成本的風險感知獎勵（如夏普比率），並對巨大未實現損失給予負面懲罰，引導代理人學會停損 。

●模型訓練與局部學習：導入 Stable Baselines3 套件，選擇 DQN 或 PPO 演算法 。

●為避免單一全域模型混淆，實施「局部學習（Local Learning）」架構，為特定特徵群組訓練專屬模型 。

### 系統回測與模型驗證
此階段的重點是防範量化回測最致命的「前視偏誤（Look-ahead Bias）」。

●轉換為事件驅動架構：從目前求快的陣列矩陣運算，全面轉換為客製化的 OpenAI Gym 事件驅動（Event-driven）環境，嚴格模擬真實市場的時間流逝、滑價與手續費摩擦成本 。

●滾動式推進分析：捨棄傳統的 K-Fold 交叉驗證，改採金融學界標準的滾動式推進分析（Walk-Forward Optimization），確保系統在 $t$ 時刻僅使用 $t$ 時刻以前的特徵 。

●消融實驗 (Ablation Study)：量化比較移除 VIX 因子前後的效能差異，以實證特徵設計的學術貢獻 。

### 論文撰寫與系統架構展現
最後階段將聚焦於收斂實證結果，並強化資訊管理領域的學術價值。

●模型可解釋性：加入 SHAP 數值分析，剖析 VIX 或價差等特徵對強化學習代理人決策的實際影響權重，提升論文深度 。

●繪製系統架構：產出涵蓋資料擷取、特徵工程、模型推論與風險控制四大層級的資訊系統軟體架構圖，突顯系統整合上的貢獻 。

●排版：依據元智大學教務處的學位論文格式規範（如 A4 規格、版面邊距、中英文摘要格式）與 APA 參考文獻標準進行最終排版定稿 。
